In [34]:
import pandas as pd

In [35]:
raw_data = pd.read_csv("./data/2022-2023IF.csv")
raw_data.drop(columns=["涨跌分值"], inplace=True)

In [48]:
# raw_data["Biggest_Change"] = raw_data[raw_data["category"].str.split(" - ")[0]]这个修改下
raw_data["Biggest_Change"] = raw_data["category"].apply(lambda x: x.split(" - ")[0])
raw_data["SCI_FI_Change"] = raw_data["category"].apply(lambda x: x.split(" - ")[1] if " - " in x else None)
raw_data["SCI_FI_Change"] = raw_data["SCI_FI_Change"].apply(lambda x: x.split("(")[0])
raw_data

,journal_name,category,if_2023,if_2022,Biggest_Change,SCI_FI_Change
0,CA-A CANCER JOURNAL FOR CLINICIANS,ONCOLOGY - SCIE(Q1),254.7,286.13,ONCOLOGY,SCIE
1,LANCET,"MEDICINE, GENERAL & INTERNAL - SCIE(Q1)",168.9,202.73,"MEDICINE, GENERAL & INTERNAL",SCIE
2,NEW ENGLAND JOURNAL OF MEDICINE,"MEDICINE, GENERAL & INTERNAL - SCIE(Q1)",158.5,176.08,"MEDICINE, GENERAL & INTERNAL",SCIE
3,JAMA-JOURNAL OF THE AMERICAN MEDICAL ASSOCIATION,"MEDICINE, GENERAL & INTERNAL - SCIE(Q1)",120.7,157.34,"MEDICINE, GENERAL & INTERNAL",SCIE
4,NATURE REVIEWS MOLECULAR CELL BIOLOGY,CELL BIOLOGY - SCIE(Q1),112.7,113.92,CELL BIOLOGY,SCIE
...,...,...,...,...,...,...
21425,FISHERY BULLETIN,FISHERIES - SCIE(Q3),0.8,1.23,FISHERIES,SCIE
21426,CESifo Economic Studies,ECONOMICS - SSCI(Q4),NaN,1.23,ECONOMICS,SSCI
21427,JOURNAL OF PSYCHOPHYSIOLOGY,"PSYCHOLOGY, BIOLOGICAL - SSCI(Q4); NEUROSCIENC...",1.3,1.23,"PSYCHOLOGY, BIOLOGICAL",SSCI
21428,MARINE AND FRESHWATER BEHAVIOUR AND PHYSIOLOGY,MARINE & FRESHWATER BIOLOGY - SCIE(Q4),1,1.23,MARINE & FRESHWATER BIOLOGY,SCIE


In [40]:
if_data_2025 = pd.read_excel("./data/2025年最新JCR完整版.xlsx", header=0)
if_data_2025

,Rank,Journal Name,JCR Year,Abbreviated Journal,Publisher,ISSN,eISSN,Total Cites,Total Articles,Citable Items,Cited Half-Life,Citing Half-Life,JIF 2024,5-Year JIF,JIF Without Self-Cites,JCI,JIF Quartile,JIF Rank
0,1,CA-A CANCER JOURNAL FOR CLINICIANS,2024,CA-CANCER J CLIN,WILEY,0007-9235,1542-4863,71799.0,6.0,14.0,3.7,4.7,232.4,353,232.2,112.16,Q1,1/326
1,2,NATURE REVIEWS MICROBIOLOGY,2024,NAT REV MICROBIOL,NATURE PORTFOLIO,1740-1526,1740-1534,61109.0,0.0,49.0,6.7,5.4,103.3,99.1,102.8,9.86,Q1,1/163
2,3,NATURE REVIEWS DRUG DISCOVERY,2024,NAT REV DRUG DISCOV,NATURE PORTFOLIO,1474-1776,1474-1784,52833.0,4.0,36.0,6.5,5.4,101.8,129.8,101.1,13.95,Q1,1/352
3,4,NATURE REVIEWS MOLECULAR CELL BIOLOGY,2024,NAT REV MOL CELL BIO,NATURE PORTFOLIO,1471-0072,1471-0080,73135.0,0.0,49.0,7.1,6.9,90.2,128.7,89.6,7.94,Q1,1/204
4,5,Kidney International Supplements,2024,KIDNEY INT SUPPL,ELSEVIER SCIENCE INC,2157-1724,2157-1716,3196.0,9.0,9.0,11.0,5.6,89.6,26,89.6,5.88,Q1,1/133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20444,20445,SCULPTURE MONUMENTS AND OPEN SPACE,2024,SCULPT MONUM OPEN SP,WILEY,2993-3439,2993-3439,1.0,11.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20445,20446,Slavic Literatures,2024,SLAV LITERATURES,ELSEVIER,2950-4244,2950-3965,1.0,29.0,29.0,NaN,29.3,NaN,NaN,NaN,NaN,NaN,NaN
20446,20447,Substance Use & Addiction Journal,2024,SUBST USE ADDICT J,SAGE PUBLICATIONS INC,2976-7342,2976-7350,74.0,109.0,116.0,NaN,5.9,NaN,NaN,NaN,NaN,NaN,NaN
20447,20448,Substance Use-Research and Treatment,2024,SUBST USE-RES TREATM,SAGE PUBLICATIONS LTD,NaN,2976-8357,5.0,20.0,23.0,NaN,6.3,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
if_data_2025_selected = if_data_2025[
    [
        "Journal Name",
        "JIF Quartile",
        "JIF 2024",
    ]
].rename(
    columns={
        "Journal Name": "journal_name",
        "JIF Quartile": "jif_quartile_2025",
        "JIF 2024": "if_2024",
    }
)

In [53]:
if_data_2025_selected

,journal_name,jif_quartile_2025,if_2024
0,CA-A CANCER JOURNAL FOR CLINICIANS,Q1,232.4
1,NATURE REVIEWS MICROBIOLOGY,Q1,103.3
2,NATURE REVIEWS DRUG DISCOVERY,Q1,101.8
3,NATURE REVIEWS MOLECULAR CELL BIOLOGY,Q1,90.2
4,Kidney International Supplements,Q1,89.6
...,...,...,...
20444,SCULPTURE MONUMENTS AND OPEN SPACE,NaN,NaN
20445,Slavic Literatures,NaN,NaN
20446,Substance Use & Addiction Journal,NaN,NaN
20447,Substance Use-Research and Treatment,NaN,NaN


In [66]:
# 为两个DataFrame创建小写的临时匹配列
if_data_2025_selected['journal_name_lower'] = if_data_2025_selected['journal_name'].str.lower()
raw_data['journal_name_lower'] = raw_data['journal_name'].str.lower()

# 左连接（保留if_data_2025所有行），基于小写列匹配
merged_df = pd.merge(
    if_data_2025_selected,
    raw_data,
    on='journal_name_lower',
    how='left',
    suffixes=('_if', '_raw')  # 区分同名列
)

# 删除临时匹配列
merged_df.drop(columns='journal_name_lower', inplace=True)

In [67]:
# merged_data找到	if_2024是na的行
missing_if_2024 = merged_df[merged_df["if_2024"].isna()]
missing_if_2024

,journal_name_if,jif_quartile_2025,if_2024,journal_name_raw,category,if_2023,if_2022,Biggest_Change,SCI_FI_Change
20389,Westminster Papers in Communication & Culture,NaN,NaN,Westminster Papers in Communication & Culture,COMMUNICATION - ESCI(N/A),NaN,NaN,COMMUNICATION,ESCI
20390,Literature Critique and Empire Today,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20391,Asian Journal of Agriculture and Biology,NaN,NaN,Asian Journal of Agriculture and Biology,"AGRICULTURE, MULTIDISCIPLINARY - ESCI(N/A)",NaN,NaN,"AGRICULTURE, MULTIDISCIPLINARY",ESCI
20392,Journal of Crop Health,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20393,MECHANICS OF ADVANCED MATERIALS AND STRUCTURES,NaN,NaN,MECHANICS OF ADVANCED MATERIALS AND STRUCTURES,"MATERIALS SCIENCE, COMPOSITES - SCIE(Q2); MATE...",2.8,3.34,"MATERIALS SCIENCE, COMPOSITES",SCIE
...,...,...,...,...,...,...,...,...,...
20445,SCULPTURE MONUMENTS AND OPEN SPACE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20446,Slavic Literatures,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20447,Substance Use & Addiction Journal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20448,Substance Use-Research and Treatment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
raw_data[raw_data["journal_name"] == "IMMUNITY"]

,journal_name,category,if_2023,if_2022,Biggest_Change,SCI_FI_Change
49,IMMUNITY,IMMUNOLOGY - SCIE(Q1),32.4,43.47,IMMUNOLOGY,SCIE


In [62]:
if_data_2025_selected[if_data_2025_selected["journal_name"] == "immunity"]

,journal_name,jif_quartile_2025,if_2024
